- ## Importing required libraries

In [0]:

from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType,DateType


In [0]:
%run "/Workspace/Users/abdulm63633@gmail.com/Ecommerce Lakehouse Project/01_setup_file/setup_utils"

In [0]:
print(bronze_schema,silver_schema,gold_schema)

In [0]:
dbutils.widgets.text("catalog", "ecommerce_lakehouse_project", "Catalog")
dbutils.widgets.text("data_source", "shipments", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

base_path = f's3://batch-ecommerce-lakehouse-project/{data_source}/landing/*.json'
print(base_path)

In [0]:
schema = StructType([
    StructField("delivery_date", DateType(), True),
    StructField("shipment_id", StringType(), True),
    StructField("shipment_status", StringType(), True),
    StructField("warehouse", StringType(), True),
])

df = (
    spark.read.format("json")
        .schema(schema)
        .option("mode", "PERMISSIVE")
        .option("columnNameOfCorruptRecord", "_corrupt_record")
        .load(base_path)
        .withColumn("read_timestamp", F.current_timestamp())
        .withColumn("batch_date", F.current_date())
        .select("*", "_metadata.file_name", "_metadata.file_size")

)
display(df.limit(10))

In [0]:
df.printSchema()

In [0]:
df.write \
    .format("delta") \
    .option("delta.enableChangeDataFeed", "true") \
    .option("mergeSchema", "true") \
    .mode("append") \
    .saveAsTable(
        f"{catalog}.{bronze_schema}.{data_source}"
    )